# Data Gathering Demos

This notebook contains standalone data-gathering examples for the Real World Data Wrangling presentation. These demos create or load data that is not used by the main cleaning and analysis notebook, so they live here instead of interrupting the project-flow notebook.

## Setup

This notebook lives in its own folder. It reads the source CSV files from the parent `Real World Data Wrangling` folder and writes temporary demo outputs into this notebook folder.

In [1]:
from pathlib import Path
import zipfile

import pandas as pd

NOTEBOOK_DIR = Path(".")
DATA_DIR = Path("..")
programs_path = DATA_DIR / "library_program_registrations_dirty.csv"
branches_path = DATA_DIR / "library_branch_lookup.csv"

print("Program dataset exists:", programs_path.exists())
print("Branch dataset exists:", branches_path.exists())

programs_raw = pd.read_csv(programs_path)
branches_raw = pd.read_csv(branches_path)
print("Program rows:", len(programs_raw))
print("Branch rows:", len(branches_raw))

Program dataset exists: True
Branch dataset exists: True
Program rows: 28
Branch rows: 7


### Files on Hand: CSV, ZIP, JSON, and Excel Notes

The source files are CSVs, but this cell also demonstrates how ZIP and JSON can fit into the same workflow. Excel files follow the same idea with `pd.read_excel("file.xlsx")`, although this demo avoids requiring the extra `openpyxl` package.

In [2]:
# Demonstrate a ZIP file workflow using one of the existing CSV datasets.
archive_path = NOTEBOOK_DIR / "demo_files_on_hand.zip"
with zipfile.ZipFile(archive_path, mode="w") as archive:
    archive.write(programs_path, arcname=programs_path.name)

with zipfile.ZipFile(archive_path) as archive:
    print("Files inside ZIP:", archive.namelist())

programs_from_zip = pd.read_csv(archive_path)
programs_from_zip.head(3)

Files inside ZIP: ['library_program_registrations_dirty.csv']


,registration_id,learner_name,Age,city,branch_id,program_id,program_name,session_date,program_type,interests,registration_status,attendance_hours,pre_score,post_score,fee_paid,scholarship_amount,empty_column
0,R001,Emma Johnson,8,New York,B101,P001,Intro to Python,2026-01-12,STEM,coding; robots,Attended,2.0,45,72,$0,$25,NaN
1,R002,Liam Smith,10,Los Angeles,B102,P001,Intro to Python,01/20/2026,STEM,coding; games,Attended,2.5,50,80,$0,$25,NaN
2,R003,Olivia Brown,11,Chicago,B103,P002,Data Storytelling,Feb 02 2026,Data,charts; storytelling,No Show,0,60,*,$5,$0,NaN


In [3]:
# Demonstrate JSON by saving and reloading the branch lookup as records.
branches_json_path = NOTEBOOK_DIR / "branch_lookup_demo.json"
branches_raw.to_json(branches_json_path, orient="records", indent=2)
branches_from_json = pd.read_json(branches_json_path)
branches_from_json.head(3)

,branch_id,branch_name,branch_city,state,region,opened_year,square_feet,annual_visits,wifi_available
0,B101,Central Library,New York,NY,Northeast,1911,80000,"1,250,000",True
1,B102,Westside Branch,Los Angeles,CA,West,1954,45000,"850,000",True
2,B103,Lakeview Branch,Chicago,IL,Midwest,1972,38000,"620,000",True


### Optional API Example

This uses a no-key US ZIP-code API endpoint to show the `requests.get()` pattern. If the network is unavailable, the notebook prints the reason and continues.

In [4]:
import requests

api_url = "https://api.zippopotam.us/us/90210"
try:
    response = requests.get(api_url, timeout=10, headers={"User-Agent": "udacity-demo"})
    response.raise_for_status()
    payload = response.json()
    zip_lookup = pd.DataFrame(payload["places"])
    zip_lookup["post_code"] = payload["post code"]
    api_preview = zip_lookup[["post_code", "place name", "state", "latitude", "longitude"]]
    print("Loaded API data:")
    print(api_preview.to_string(index=False))
except Exception as exc:
    zip_lookup = pd.DataFrame()
    print("API demo skipped:", exc)

Loaded API data:
post_code    place name      state latitude longitude
    90210 Beverly Hills California  34.0901 -118.4065


### Optional Kaggle API Example

For the Kaggle slide, use a non-movie topic so learners can still choose the movie dataset for their own project.

Suggested dataset: `camnugent/california-housing-prices`

Topic: California housing prices from census-based district data.

This public dataset can be downloaded directly from Kaggle's dataset API endpoint without a CLI or API token. The cell reuses the local copy after the first successful download.

In [5]:
kaggle_dataset = "camnugent/california-housing-prices"
kaggle_dir = NOTEBOOK_DIR / "kaggle_california_housing"
kaggle_archive = kaggle_dir / "california-housing-prices.zip"
kaggle_file = kaggle_dir / "housing.csv"
kaggle_api_url = f"https://www.kaggle.com/api/v1/datasets/download/{kaggle_dataset}"

print("Kaggle dataset:", kaggle_dataset)

if not kaggle_file.exists():
    kaggle_dir.mkdir(exist_ok=True)
    kaggle_response = requests.get(
        kaggle_api_url, timeout=60, headers={"User-Agent": "udacity-demo"}
    )
    kaggle_response.raise_for_status()
    kaggle_archive.write_bytes(kaggle_response.content)
    with zipfile.ZipFile(kaggle_archive) as archive:
        archive.extract("housing.csv", path=kaggle_dir)
    print("Downloaded data from the Kaggle API.")
else:
    print("Using the existing Kaggle download.")

housing = pd.read_csv(kaggle_file)
print("Loaded Kaggle data:", housing.shape)
print(housing.head(3).to_string(index=False))

Kaggle dataset: camnugent/california-housing-prices


Downloaded data from the Kaggle API.
Loaded Kaggle data: (20640, 10)
 longitude  latitude  housing_median_age  total_rooms  total_bedrooms  population  households  median_income  median_house_value ocean_proximity
   -122.23     37.88                41.0        880.0           129.0       322.0       126.0         8.3252            452600.0        NEAR BAY
   -122.22     37.86                21.0       7099.0          1106.0      2401.0      1138.0         8.3014            358500.0        NEAR BAY
   -122.24     37.85                52.0       1467.0           190.0       496.0       177.0         7.2574            352100.0        NEAR BAY
